# InternCircle Data Science & AI — Task 1
## Exploratory Data Analysis (EDA) on the Titanic Dataset

**Intern:** Tina Devi  
**Internship:** Data Science & AI  
**InternCircle ID:** IC-2026-1286  
**Task:** Exploratory Data Analysis (EDA) on Titanic  
**Objective:** Use Pandas to inspect, clean, summarize, and analyze the Titanic dataset.

### Task requirements
- DataFrame operations: `head()`, `info()`, `describe()`
- Handle missing values and duplicates
- Perform grouping and aggregation
- Extract useful statistical insights

> This notebook is designed as a beginner-friendly, submission-ready internship project.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


## 1. Load the Dataset

In [ ]:
# Load the local Titanic dataset included in this repository
df = pd.read_csv("../data/titanic.csv")

print(f"Dataset shape: {df.shape}")
df.head()


## 2. Understand the Dataset

In [ ]:
# First five rows
df.head()


In [ ]:
# Dataset structure, data types, and non-null counts
df.info()


In [ ]:
# Descriptive statistics for numerical columns
df.describe()


## 3. Check Columns and Basic Statistics

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nNumber of rows:", len(df))
print("Number of columns:", df.shape[1])

print("\nUnique values:")
display(df.nunique().sort_values(ascending=False).to_frame("unique_count"))


## 4. Check Missing Values

In [ ]:
missing = (
    df.isna()
      .sum()
      .to_frame("missing_values")
      .assign(missing_percentage=lambda x: x["missing_values"] / len(df) * 100)
      .sort_values("missing_values", ascending=False)
)

missing


### Missing-value interpretation

The main missing-value issues are in **Cabin** and **Age**, while **Embarked** has only a small number of missing observations. For this EDA, the analysis dataset will be cleaned by:
- filling missing `Age` values with the median age;
- filling missing `Embarked` values with the most frequent category;
- treating missing `Cabin` as `"Unknown"` so that the information is retained as a category.


## 5. Check for Duplicate Rows

In [ ]:
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)


## 6. Clean the Dataset

In [ ]:
clean_df = df.copy()

# Fill missing numerical values with the median
clean_df["Age"] = clean_df["Age"].fillna(clean_df["Age"].median())

# Fill missing categorical values with the mode
clean_df["Embarked"] = clean_df["Embarked"].fillna(clean_df["Embarked"].mode()[0])

# Keep missing cabin information as an explicit category
clean_df["Cabin"] = clean_df["Cabin"].fillna("Unknown")

# Remove exact duplicate rows, if any
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

print("Shape before cleaning:", df.shape)
print("Shape after cleaning :", clean_df.shape)
print("\nRemaining missing values:")
display(clean_df.isna().sum().to_frame("missing_values"))


## 7. Overall Survival Analysis

In [ ]:
survival_counts = clean_df["Survived"].value_counts().sort_index()
survival_rate = clean_df["Survived"].mean() * 100

print(f"Overall survival rate: {survival_rate:.2f}%")
print("\nSurvival counts:")
print(survival_counts.rename(index={0: "Did not survive", 1: "Survived"}))


## 8. Survival by Gender

In [ ]:
gender_analysis = (
    clean_df.groupby("Sex")["Survived"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "passengers", "sum": "survivors", "mean": "survival_rate"})
)

gender_analysis["survival_rate"] *= 100
gender_analysis.sort_values("survival_rate", ascending=False)


## 9. Survival by Passenger Class

In [ ]:
class_analysis = (
    clean_df.groupby("Pclass")["Survived"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "passengers", "sum": "survivors", "mean": "survival_rate"})
)

class_analysis["survival_rate"] *= 100
class_analysis


## 10. Survival by Gender and Passenger Class

In [ ]:
gender_class_analysis = (
    clean_df.groupby(["Pclass", "Sex"])["Survived"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "passengers", "sum": "survivors", "mean": "survival_rate"})
)

gender_class_analysis["survival_rate"] *= 100
gender_class_analysis


## 11. Age Group Analysis

In [ ]:
clean_df["AgeGroup"] = pd.cut(
    clean_df["Age"],
    bins=[0, 12, 18, 35, 60, np.inf],
    labels=["Child", "Teenager", "Young Adult", "Adult", "Senior"],
    include_lowest=True
)

age_group_analysis = (
    clean_df.groupby("AgeGroup", observed=True)["Survived"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "passengers", "sum": "survivors", "mean": "survival_rate"})
)

age_group_analysis["survival_rate"] *= 100
age_group_analysis


## 12. Family Size Analysis

In [ ]:
clean_df["FamilySize"] = clean_df["SibSp"] + clean_df["Parch"] + 1

family_analysis = (
    clean_df.groupby("FamilySize")["Survived"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "passengers", "sum": "survivors", "mean": "survival_rate"})
)

family_analysis["survival_rate"] *= 100
family_analysis


## 13. Fare and Age Summary by Survival

In [ ]:
survival_numeric_summary = (
    clean_df.groupby("Survived")[["Age", "Fare", "FamilySize"]]
    .mean()
    .rename(index={0: "Did not survive", 1: "Survived"})
)

survival_numeric_summary


## 14. Useful EDA Visualizations

In [ ]:
# Survival distribution
plt.figure(figsize=(6, 4))
clean_df["Survived"].map({0: "Did not survive", 1: "Survived"}).value_counts().plot(kind="bar")
plt.title("Titanic Survival Distribution")
plt.xlabel("Outcome")
plt.ylabel("Number of passengers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Survival rate by passenger class
class_analysis["survival_rate"].plot(kind="bar", figsize=(6, 4))
plt.title("Survival Rate by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Survival Rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Survival rate by gender
gender_analysis["survival_rate"].plot(kind="bar", figsize=(6, 4))
plt.title("Survival Rate by Gender")
plt.xlabel("Gender")
plt.ylabel("Survival Rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 15. Key Findings

In [ ]:
overall_rate = clean_df["Survived"].mean() * 100
female_rate = clean_df.loc[clean_df["Sex"] == "female", "Survived"].mean() * 100
male_rate = clean_df.loc[clean_df["Sex"] == "male", "Survived"].mean() * 100
first_rate = clean_df.loc[clean_df["Pclass"] == 1, "Survived"].mean() * 100
second_rate = clean_df.loc[clean_df["Pclass"] == 2, "Survived"].mean() * 100
third_rate = clean_df.loc[clean_df["Pclass"] == 3, "Survived"].mean() * 100

print(f"1. Overall survival rate: {overall_rate:.2f}%")
print(f"2. Female survival rate: {female_rate:.2f}%")
print(f"3. Male survival rate: {male_rate:.2f}%")
print(f"4. 1st-class survival rate: {first_rate:.2f}%")
print(f"5. 2nd-class survival rate: {second_rate:.2f}%")
print(f"6. 3rd-class survival rate: {third_rate:.2f}%")


### Conclusion

The exploratory analysis shows clear differences in Titanic survival outcomes across passenger characteristics. Survival rates vary substantially by **gender** and **passenger class**, while age and family size also provide useful segmentation variables.

This task demonstrates practical use of **Pandas** for:
- loading and inspecting a dataset;
- identifying and handling missing data;
- checking duplicates;
- generating descriptive statistics;
- grouping and aggregating data;
- extracting business/data insights from structured data.

**Next step:** Task 2 — Data Visualization with Matplotlib & Seaborn.
